In [0]:
fact_medications = spark.sql(f"select * from regis_healthcare.silver.medications;")
fact_medications.createOrReplaceTempView("medications")

In [0]:
# Fact_Medications --> Source: medications
# | Foreign Keys        |
# | ------------------- |
# | medication_fact_key |
# | resident_key        |
# | medication_key      |
# | start_date_key      |
# | end_date_key        |
# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number

# window_spec = Window.orderBy("medication_id")

# fact_medications = fact_medications.withColumn(
#     "medication_fact_key",
#     row_number().over(window_spec)
# )
fact_medications = spark.sql("""SELECT 
    *,
    ROW_NUMBER() OVER (ORDER BY medication_id) AS medication_fact_key
FROM medications;
""")
#---- existing table id
# from pyspark.sql.functions import max

# max_key = df_medications.agg(
#     max("medication_fact_key")
# ).collect()[0][0]

# max_key = max_key if max_key else 0

# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number

# window_spec = Window.orderBy("medication_id")

# df_medications = df_medications.withColumn(
#     "medication_fact_key",
#     row_number().over(window_spec) + max_key
# )
# display(df_medications)
#-------------------------------
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_medications = fact_medications.withColumn("start_date_key", date_format(col("start_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_medications = fact_medications.withColumn("start_date_key", col("start_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_medications = fact_medications.withColumn("end_date_key", date_format(col("end_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_medications = fact_medications.withColumn("end_date_key", col("end_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
fact_medications = fact_medications.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_medications = fact_medications.withColumn(
    "medication_key",
    regexp_replace(col("medication_id"), "^MED", "").cast("int")
)
# display(df_facilities)
fact_medications = fact_medications.select(
 "medication_fact_key", 
 "resident_key",        
 "medication_key",      
 "start_date_key",     
 "end_date_key"   
)
display(fact_medications)



#### cataloge 

In [0]:
fact_medications.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_medications")

In [0]:
fact_medications.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_medications")
print(fact_medications.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_medications")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_medications")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.medication_key = source.medication_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_medications;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_medications;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_medications.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_medications")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_medications"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_medications

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.medication_key = source.medication_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
